In [ ]:
#installing dependencies and libraries
!pip install -q tifffile scikit-image scikit-learn seaborn rasterio

In [ ]:
import os, warnings
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tifffile
from pathlib import Path
from PIL import Image, ImageStat
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (classification_report, confusion_matrix, roc_curve, auc, roc_auc_score,ConfusionMatrixDisplay, accuracy_score)
from skimage.feature import graycomatrix, graycoprops
from skimage.transform import resize as sk_resize

directory = Path('PhDMangroveDataset')
mangrove_class = ['mangroves', 'nonmangroves']
labbeled_class = {'mangroves': 1, 'nonmangroves': 0}
zero_value = 0.0
seed = 42
image_shape = (256, 256, 7)

**This section is pre-processing of the mangrove dataset from @Kaggle**

In [ ]:
for cls in mangrove_class:
    folder = directory / cls

def corruption(path: str) -> str:
    try:
        #this is reading the file using tifffile (originally i used the OS method after i couldnt get tifffile to work however i reversed engineered someones kaggle code on this dataset to learn) 
        data = tifffile.imread(path).astype(np.float32)
        if data.ndim == 2:
            data = data[..., np.newaxis] #this is adding a new axis so it goes from 2d to 3d + colour
        elif data.shape[0] < data.shape[-1]:
            data = data.transpose(1, 2, 0) #transposing the data so it goes from (bands, height, width) to (height, width, bands) 256x256x7
        valid = data[data != zero_value]
        if valid.size == 0:           return 'dark' 
        mean = float(valid.mean())
        std  = float(valid.std())
        if mean < 0.01:               return 'dark'
        if std < 0.001:               return 'uniform'
        return 'valid'
    except Exception as e:
        return f'error: {e}'

print(f"{'Class':<20} {'valid':>8} {'dark':>8} {'uniform':>10} {'error':>8}")


#this is going through every image in the dataset and is calling the curruption function to determine if it is valid or not
summary = {}
for cls in mangrove_class:
    folder = directory / cls
    counts = {'valid': 0, 'dark': 0, 'uniform': 0, 'error': 0}
    for tif in folder.glob('*.tif'):
        tag = corruption(str(tif))
        key = tag if tag in counts else 'error'
        counts[key] += 1
    summary[cls] = counts
    print(f"{cls:<20} {counts['valid']:>8,} {counts['dark']:>8,} "
          f"{counts['uniform']:>10,} {counts['error']:>8,}")
# this is printing out a summary of the corruption types for each class in a nice format


def preview(path, size=(128, 128)):
    try:
        data = tifffile.imread(path).astype(np.float32)
        if data.ndim == 2:
            data = np.stack([data] * 3, axis=-1) #same explaination as how we originally read the data
        elif data.shape[0] < data.shape[-1]: 
            data = data.transpose(1, 2, 0) 

        rgb = data[:, :, :3] #taking the first 3 bands as RGB 
        out = np.zeros((*rgb.shape[:2], 3), dtype=np.float32)#creating an empty array to store the normalized RGB values
        for c in range(3):
            lo, hi = np.percentile(rgb[:, :, c], [2, 98])
            if hi > lo:
                out[:, :, c] = np.clip((rgb[:, :, c] - lo) / (hi - lo), 0, 1)
        return sk_resize(out, size, anti_aliasing=True) #resizing the image to the specified size
    except Exception as e:
        return np.zeros((*size, 3))

n_cols = 12
n_rows = 12

for cls in mangrove_class:
    folder  = directory / cls
    tifs    = sorted(folder.glob('*.tif'))
    n_show  = min(n_cols * n_rows, len(tifs))

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 2, n_rows * 2))
    fig.suptitle(f'Class: {cls}  ({len(tifs)} total — showing {n_show})', fontsize=13, y=1.01)

    for i, ax in enumerate(axes.flat): #showing a preview of images in a 12x12
        if i < n_show:
            img  = preview(str(tifs[i]))
            name = tifs[i].stem
            tag  = corruption(str(tifs[i]))
            ax.imshow(img)
            color = 'red' if tag != 'valid' else 'white'
            ax.set_title(f'{name[:10]}\n{tag}', fontsize=6, color=color)
        else:
            ax.axis('off')
        ax.set_xticks([])
        ax.set_yticks([])

    plt.tight_layout()
    plt.show()

**Further pre-processing however, only keeping valid images**

In [ ]:
#this section of code is essentially going through every image and deciding which image is valido or not in mangroves and nonmangroves and printing the valid and invalid of each class
print('Determining which images are non valid')
valid_paths, valid_labels = [], [] #this is creating empty lists to store the paths and labels of the valid images
removed = {cls: 0 for cls in mangrove_class}
for path, label in zip(valid_paths, valid_labels):
    tag = corruption(path) #determing if the file is valid or not
    cls = [k for k, v in labbeled_class.items() if v == label][0] #this is finding the class name based on the label (1 for mangroves and 0 for non-mangroves)
    if tag == 'valid':
        valid_paths.append(path)
        valid_labels.append(label)
    else:
        removed[cls] += 1
valid_paths  = np.array(valid_paths)
valid_labels = np.array(valid_labels, dtype=np.int32)

print(f'\nRemoved:')
for cls, count in removed.items():#this is printing out the number of corrupt images removed for each class
    print(f'  {cls}: {count} corrupt images removed')
total = len(valid_labels)
n_pos = int(valid_labels.sum())
n_neg = total - n_pos
print(f'\nNew dataset: {total} images ({n_pos} mangrove, {n_neg} non-mangrove)')

print('Filtering corrupted images')
valid_paths, valid_labels = [], []
removed = {cls: 0 for cls in mangrove_class}
for cls in mangrove_class:
    folder = directory / cls
    tifs   = sorted(folder.glob('*.tif'))
    for tif in tifs:
        tag = corruption(str(tif))
        if tag == 'valid':
            valid_paths.append(str(tif))
            valid_labels.append(labbeled_class[cls])
        else:
            removed[cls] += 1
valid_paths  = np.array(valid_paths)
all_labels = np.array(valid_labels, dtype=np.int32)

print(f'\nRemoved:')
for cls, count in removed.items():
    print(f'  {cls}: {count} corrupt images removed')
total = len(all_labels)
n_pos = int(all_labels.sum())
n_neg = total - n_pos
ratio = n_pos / n_neg if n_neg else float('inf')
print(f'Mangrove  (1) : {n_pos:,}  ({100 * n_pos / total:.1f} %)')
print(f'Non-mang. (0) : {n_neg:,}  ({100 * n_neg / total:.1f} %)')

**Training split of the data 70% training, 15% validation set 15% testing**

In [ ]:
#Splitting the data insto training, validation and testing using stratified, lastly the split is 70% training, 15% validation and 15% testing
X_train_paths, X_tmp, y_train, y_tmp = train_test_split(valid_paths, all_labels, test_size=0.30, stratify=all_labels, random_state=seed)
X_val_paths, X_test_paths, y_val, y_test = train_test_split(X_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=seed)

**Extracting key features of the dataset, statistical and glcm**

In [ ]:
def extracting_statistical_features(band, no_data=0.0): #this is extracting statistical features from the image band such as mean, std, min, max, percentiles and IQR
    valid = band[band != no_data]
    if valid.size == 0:
        return [0.0] * 8
    p25, median, p75 = np.percentile(valid, [25, 50, 75])
    return [float(valid.mean()), float(valid.std()), float(valid.min()), float(valid.max()),p25, median, p75, float(p75 - p25)]

def extracting_glcm_features(band, no_data=0.0, n_levels=32): #this is extracting texture features GLCM (Gray Level Co-occurrence Matrix) features from the image band
    valid_mask = band != no_data
    if valid_mask.sum() == 0:
        return [0.0] * 6
    b_min, b_max = band[valid_mask].min(), band[valid_mask].max() #
    if b_max == b_min:
        return [0.0] * 6
    normalized = np.zeros_like(band, dtype=np.uint8)
    normalized[valid_mask] = (
        (band[valid_mask] - b_min) / (b_max - b_min) * (n_levels - 1)
    ).astype(np.uint8)
    glcm = graycomatrix(
        normalized, distances=[1],
        angles=[0, np.pi/4, np.pi/2, 3*np.pi/4],
        levels=n_levels, symmetric=True, normed=True
    ) #calcuating the GLCM matrix for the normalized band with specified distances and angles, levels, symmetric and normed parameters
    props = ['contrast', 'dissimilarity', 'homogeneity', 'energy', 'correlation', 'ASM']  #these are the 6 GLCM properties we are extracting
    return [float(graycoprops(glcm, p).mean()) for p in props] #this is calculating the mean of each GLCM property

def extract_from_path(tif_path: str):
    try:
        data = tifffile.imread(tif_path).astype(np.float32) #same reading  annotation as described before
        if data.ndim == 2:
            data = data[..., np.newaxis]
        elif data.shape[0] < data.shape[-1]:
            data = data.transpose(1, 2, 0)
        features = []
        for c in range(data.shape[-1]):
            band = data[:, :, c]
            features.extend(extracting_statistical_features(band, zero_value)) #this is extracting the statistical features for each band and adding them to the features list
            features.extend(extracting_glcm_features(band, zero_value))  #this is extracting the GLCM features for each band and adding them to the features list
        return np.array(features, dtype=np.float32) #this is returning the features as a numpy array of type float32
    except Exception as e:
        print(f'Could not read {tif_path}: {e}') 
        return None

def loading_of_features(paths, labels, desc='', expected_len=None): #this is loading the features from the paths and labels and is also padding the features to the same length if expected_len is provided
    X, y, lengths = [], [], []
    for i, (path, label) in enumerate(zip(paths, labels)): 
        if i % 100 == 0:
            print(f'  {desc}: {i}/{len(paths)}', end='\r')
        feats = extract_from_path(path)
        if feats is not None:
            X.append(feats)
            y.append(label)
            lengths.append(len(feats))
    print(f'  {desc}: {len(X)}/{len(paths)} loaded')
    print(f'  Feature lengths seen: {sorted(set(lengths))}')

    #Pad all vectors to the same length
    max_len = expected_len or max(lengths) #determing maximum lenght of featrures to pad to
    X_padded = np.zeros((len(X), max_len), dtype=np.float32)
    for i, feats in enumerate(X):
        X_padded[i, :len(feats)] = feats

    return X_padded, np.array(y), max_len

X_train, y_train, max_len = loading_of_features(X_train_paths, y_train, 'Train')
X_val,   y_val,   _       = loading_of_features(X_val_paths,   y_val,   'Val  ', max_len)
X_test,  y_test,  _       = loading_of_features(X_test_paths,  y_test,  'Test ', max_len)

**Random Forest Pipeline (ML Model)**

In [ ]:
#validation set merge with the training set
X_train_full = np.vstack([X_train, X_val])
y_train_full = np.concatenate([y_train, y_val])

#randomforest model pipline
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('rf', RandomForestClassifier(
        n_estimators=200,
        max_depth=None,
        min_samples_leaf=4,
        max_features='sqrt',
        class_weight='balanced',
        random_state=seed,
        n_jobs=-1,
        verbose=1))
])
print('Fitting Random Forest')
pipeline.fit(X_train_full, y_train_full)